# Matmul Operation Profiler Analysis

This notebook analyzes matmul operation performance from tracy profiler output.


In [ ]:
# Set the path to your profiler CSV file
CSV_FILE = "/home/zbaczewski/tt-metal/generated/profiler/reports/YOUR_TIMESTAMP/ops_perf_results_YOUR_TIMESTAMP.csv"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# Load the CSV file
df = pd.read_csv(CSV_FILE)

# Rename columns for clarity
df = df.rename(columns={
    'OP CODE': 'Operation',
    'HOST DURATION [ns]': 'Host Time',
    'OP TO OP LATENCY [ns]': 'Time Between Ops',
    'DEVICE FW DURATION [ns]': 'Device Time'
})

print(f"Total rows in CSV: {len(df)}")
print(f"Unique operations: {df['Operation'].unique()}")


In [ ]:
# Filter for matmul operations (case-insensitive match)
matmul_df = df[df['Operation'].str.contains('matmul', case=False, na=False)].copy()

print(f"Found {len(matmul_df)} matmul operations")
print(f"Matmul operation types: {matmul_df['Operation'].unique()}")


In [ ]:
# Convert from nanoseconds to milliseconds and microseconds
matmul_df['Host Time (ms)'] = matmul_df['Host Time'] / 1_000_000
matmul_df['Time Between Ops (ms)'] = matmul_df['Time Between Ops'] / 1_000_000
matmul_df['Device Time (ms)'] = matmul_df['Device Time'] / 1_000_000

matmul_df['Host Time (us)'] = matmul_df['Host Time'] / 1_000
matmul_df['Device Time (us)'] = matmul_df['Device Time'] / 1_000


## Summary Statistics


In [ ]:
def print_statistics(series, name, unit='ms'):
    """Print comprehensive statistics for a time series."""
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"  Count:              {len(series):>12}")
    print(f"  Mean:               {series.mean():>12.4f} {unit}")
    print(f"  Median:             {series.median():>12.4f} {unit}")
    print(f"  Std Dev:            {series.std():>12.4f} {unit}")
    print(f"  Min:                {series.min():>12.4f} {unit}")
    print(f"  Max:                {series.max():>12.4f} {unit}")
    print(f"  Range:              {series.max() - series.min():>12.4f} {unit}")
    print(f"  " + "-"*40)
    print(f"  Percentiles:")
    print(f"    5th:              {series.quantile(0.05):>12.4f} {unit}")
    print(f"    25th (Q1):        {series.quantile(0.25):>12.4f} {unit}")
    print(f"    50th (Median):    {series.quantile(0.50):>12.4f} {unit}")
    print(f"    75th (Q3):        {series.quantile(0.75):>12.4f} {unit}")
    print(f"    90th:             {series.quantile(0.90):>12.4f} {unit}")
    print(f"    95th:             {series.quantile(0.95):>12.4f} {unit}")
    print(f"    99th:             {series.quantile(0.99):>12.4f} {unit}")
    print(f"  " + "-"*40)
    print(f"  Total:              {series.sum():>12.4f} {unit}")
    iqr = series.quantile(0.75) - series.quantile(0.25)
    print(f"  IQR:                {iqr:>12.4f} {unit}")
    cv = (series.std() / series.mean()) * 100 if series.mean() != 0 else 0
    print(f"  Coeff of Variation: {cv:>12.2f} %")


In [ ]:
# Device Time Statistics (most important for matmul performance)
print_statistics(matmul_df['Device Time (ms)'], 'DEVICE TIME (Kernel Execution)', 'ms')
print_statistics(matmul_df['Device Time (us)'], 'DEVICE TIME (Kernel Execution)', 'us')


In [ ]:
# Host Time Statistics
print_statistics(matmul_df['Host Time (ms)'], 'HOST TIME (End-to-End)', 'ms')


In [ ]:
# Op-to-Op Latency Statistics
print_statistics(matmul_df['Time Between Ops (ms)'], 'OP-TO-OP LATENCY', 'ms')


## Visualizations


In [ ]:
# Histogram of Device Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(matmul_df['Device Time (ms)'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(matmul_df['Device Time (ms)'].mean(), color='red', linestyle='--', label=f'Mean: {matmul_df["Device Time (ms)"].mean():.4f} ms')
axes[0].axvline(matmul_df['Device Time (ms)'].median(), color='orange', linestyle='--', label=f'Median: {matmul_df["Device Time (ms)"].median():.4f} ms')
axes[0].set_xlabel('Device Time (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Matmul Device Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(matmul_df['Device Time (ms)'].dropna(), vert=True)
axes[1].set_ylabel('Device Time (ms)')
axes[1].set_title('Box Plot of Matmul Device Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Time series plot of all matmul operations
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(range(len(matmul_df)), matmul_df['Device Time (ms)'].values, marker='o', markersize=4, color='steelblue', alpha=0.7)
ax.axhline(matmul_df['Device Time (ms)'].mean(), color='red', linestyle='--', label=f'Mean: {matmul_df["Device Time (ms)"].mean():.4f} ms')
ax.fill_between(range(len(matmul_df)), 
                matmul_df['Device Time (ms)'].mean() - matmul_df['Device Time (ms)'].std(),
                matmul_df['Device Time (ms)'].mean() + matmul_df['Device Time (ms)'].std(),
                alpha=0.2, color='red', label='±1 Std Dev')
ax.set_xlabel('Matmul Invocation #')
ax.set_ylabel('Device Time (ms)')
ax.set_title('Matmul Device Time Over Invocations')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Compare Host vs Device Time
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Device Time', 'Host Time', 'Time Between Ops']
means = [matmul_df['Device Time (ms)'].mean(), 
         matmul_df['Host Time (ms)'].mean(), 
         matmul_df['Time Between Ops (ms)'].mean()]
stds = [matmul_df['Device Time (ms)'].std(), 
        matmul_df['Host Time (ms)'].std(), 
        matmul_df['Time Between Ops (ms)'].std()]

bars = ax.bar(metrics, means, yerr=stds, capsize=5, color=['steelblue', 'coral', 'seagreen'], alpha=0.7)
ax.set_ylabel('Time (ms)')
ax.set_title('Average Time Breakdown for Matmul Operations')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{mean:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


## Throughput Analysis


In [ ]:
# Calculate throughput metrics
# Matmul dimensions from the example: M=16384, K=1536, N=384
M, K, N = 16384, 1536, 384

# FLOPs for matmul: 2 * M * N * K (multiply-add)
flops_per_matmul = 2 * M * N * K

print(f"Matmul Dimensions: [{M}, {K}] x [{K}, {N}] = [{M}, {N}]")
print(f"FLOPs per matmul: {flops_per_matmul:,} ({flops_per_matmul/1e9:.3f} GFLOPs)")
print()

# Calculate TFLOPS based on device time
device_time_sec = matmul_df['Device Time (ms)'].mean() / 1000
if device_time_sec > 0:
    tflops = (flops_per_matmul / device_time_sec) / 1e12
    print(f"Average Device Time: {device_time_sec*1000:.4f} ms")
    print(f"Estimated Throughput: {tflops:.3f} TFLOPS")
    print()
    
    # Min/Max throughput
    min_time_sec = matmul_df['Device Time (ms)'].min() / 1000
    max_time_sec = matmul_df['Device Time (ms)'].max() / 1000
    if min_time_sec > 0:
        max_tflops = (flops_per_matmul / min_time_sec) / 1e12
        print(f"Peak Throughput (fastest run): {max_tflops:.3f} TFLOPS")
    if max_time_sec > 0:
        min_tflops = (flops_per_matmul / max_time_sec) / 1e12
        print(f"Min Throughput (slowest run): {min_tflops:.3f} TFLOPS")


## Raw Data


In [ ]:
# Display the matmul data
display_cols = ['Operation', 'Device Time (ms)', 'Host Time (ms)', 'Time Between Ops (ms)']
if 'CORE COUNT' in matmul_df.columns:
    display_cols.append('CORE COUNT')
if 'ATTRIBUTES' in matmul_df.columns:
    display_cols.append('ATTRIBUTES')

matmul_df[display_cols]


In [ ]:
# Pandas built-in describe for quick reference
matmul_df[['Device Time (ms)', 'Host Time (ms)', 'Time Between Ops (ms)']].describe()
